[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/seismic_waves/seismic_waves.ipynb)

# Seismic Waves

This example computes the elastic waves that a shallow oblique impact sends through a planet modelled as a liquid core inside a solid shell with a free surface. Compressional P waves travel fastest through solid and liquid alike, slowing and refracting toward a focus inside the denser core. Shear S waves travel slower and cannot propagate through a liquid, so the core casts a shear shadow that S waves reach only by going around through the shell. A Rayleigh wave rolls along the free surface without spreading into the interior and soon carries most of the motion across the plane-strain cross-section.

The term ***simplex gradient*** here means the plane `sides / (2 * patches)` obtained by dividing each corner's opposite edge by twice the triangle's oriented patch, so pairing `gradients & displacement` reads a corner displacement's directional rate of rise across the triangle. Binding each triangle's three simplex gradients into a symmetric work form on gradient pairs pulls the constitutive law back along corner incidences to a nodal stiffness `(Force+ <- Point-)[V <- V]`, while setting the shear modulus to zero and applying the deviatoric shear form to the velocity gives the core a slight shear viscosity without resisting swelling. Pairing the simplex gradients with nodal velocities via `gradients & Point` measures metric-free dilation rate, whereas wedging `gradients ^ Point.dual_inverse()` measures infinitesimal rotation.

In Helmholtz-potential and matrix finite-element notation, the displacement splits as $\mathbf u = \nabla\phi + \nabla \times \boldsymbol\psi$ with wave speeds $\alpha = \sqrt{(\lambda + 2\mu)/\rho}$ and $\beta = \sqrt{\mu/\rho}$, while each triangle requires $2 \times 6$ Voigt strain-displacement matrices $B_e$ built from inverted $2 \times 2$ coordinate Jacobians, $3 \times 3$ constitutive matrices $D_e$ for engineering shear strains $\gamma_{xy} = \partial_y u_x + \partial_x u_y$, and scalar divergence and out-of-plane curl stencils $\partial_x \dot u_x + \partial_y \dot u_y$ and $\partial_x \dot u_y - \partial_y \dot u_x$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import chain, repeat
import numpy as np

from numga import NumpyContext, stack, where
from numga.algebras import VGA2D as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.mechanics.seismic_waves import render, scenarios
from examples.util.animation import save_figure, show_animation, show_states
from examples.util.shapes import concentric_disk

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                      # Scalar+
Point = ga.gatype.antivector()                                   # Antivector-
Plane = ga.gatype.vector()                                       # Vector+
Force = ga.gatype.vector()                                       # Vector+
Bivector = ga.gatype.bivector()                                  # Bivector+
Acceleration = ga.gatype((Point, Point))                         # Point- <- Point-
Stiffness = ga.gatype((Force, Point))                            # Force+ <- Point-
Divergence = ga.gatype((Scalar, Point))                          # Scalar+ <- Point-
Rotation = ga.gatype((Bivector, Point))                          # Bivector+ <- Point-
Elastic = ga.gatype((Scalar, Plane, Plane, Point, Point))        # Scalar+ <- (Plane+, Plane+, Point-, Point-)

antiscalar = mv.xy.dual()                                        # Antiscalar-
dilation_traction = Plane * (Plane & Point)                      # [] Plane+ <- (Plane+, Plane+, Point-)
# Subtracting the exterior-product term removes infinitesimal rotation, leaving symmetric shear.
direct_shear = (Plane | Plane) * Point.dual_inverse()            # [] Plane+ <- (Plane+, Plane+, Point-)
shear_traction = 2 * direct_shear - (Plane | (Plane ^ Point.dual_inverse()))  # [] Plane+ <- (Plane+, Plane+, Point-)

## Liquid core and solid shell

The cross-section is a single triangulated disk partitioned into a liquid core and a solid shell whose outer rim is the free surface. Liquid and solid share the vertices along the core-mantle boundary, where slight liquid viscosity couples tangential motion across the interface. Because the core is five times denser than the shell and carries P waves more slowly, its compressional impedance is roughly three times higher, so incident waves partially reflect and partially refract into it.

In [ ]:
topology, coordinates = concentric_disk(scenarios.radial_boundaries, scenarios.n_radial_divisions)
vertices = mv.vector(coordinates).dual().field()                 # Point-[V]
centres = topology.incidence[2, 0].mean(axis=0) * vertices       # Point-[F]
# Region zero is the liquid core; region one is the solid shell.
regions = (centres.antinorm() > scenarios.core_radius).astype(int)  # [F]

density = mv.scalar(scenarios.layer_densities[regions, None]).field()          # Scalar+[F]
lame_lambda = mv.scalar(scenarios.layer_lambdas[regions, None]).field()        # Scalar+[F]
lame_mu = mv.scalar(scenarios.layer_mus[regions, None]).field()                # Scalar+[F]
viscosity_field = mv.scalar(scenarios.layer_viscosities[regions, None]).field()  # Scalar+[F]

save_figure(render.materials_figure(topology, vertices, regions), "seismic_waves_planet_materials");

Warm cream triangles mark the solid shell and pale blue triangles mark the liquid core, with dark outlines along the free surface and the core-mantle boundary.

## Strain energy, stiffness, and viscous drag

Each triangle's three simplex gradients give a constant strain across the simplex. The solid shell stores elastic energy in both dilation and symmetric shear, whereas the liquid core stores energy only in dilation and resists shear rates through a deviatoric viscous form that leaves pure swelling undamped. Sharing one third of each triangle's mass with each of its corners turns nodal forces into accelerations, while rigid translations and infinitesimal turns cost zero elastic or viscous work.

In continuum and finite-element notation, these constant-strain triangles integrate $\tfrac12 \int \boldsymbol\varepsilon(\mathbf u) : \mathbb C : \boldsymbol\varepsilon(\mathbf u)\,dA$ with $\boldsymbol\varepsilon(\mathbf u) = \tfrac12(\nabla\mathbf u + \nabla\mathbf u^{\mathsf T})$, and the liquid stress reads $\boldsymbol\sigma = \lambda\,(\nabla\cdot\mathbf u)\,I + 2\eta\,\big(\dot{\boldsymbol\varepsilon} - \tfrac13(\nabla\cdot\dot{\mathbf u})\,I\big)$.

In [ ]:
def triangle_areas(topology: Topology, vertices: Point) -> Scalar:
    """Oriented corner displacements join into each triangle's area (`Scalar+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)          # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                   # Patch-[F]
    return patches.antinorm()                                             # Scalar+[F]


def simplex_gradients(topology: Topology, vertices: Point) -> Plane:
    """Each triangle's opposite edge divided by twice its oriented patch gives its three simplex gradients (`[3] Plane+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)          # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                   # Patch-[F]
    return sides / (2 * patches)                                          # [3] Plane+[F]


areas = triangle_areas(topology, vertices)                                # Scalar+[F]
gradients = simplex_gradients(topology, vertices)                         # [3] Plane+[F]

In [ ]:
def constitutive_form(lame_lambda: Scalar, lame_mu: Scalar) -> Elastic:
    """Plane-strain isotropic Lamé parameters define a symmetric elastic work form on pairs of displacement gradients."""
    traction = lame_lambda * dilation_traction + lame_mu * shear_traction  # [...] Plane+ <- (Plane+, Plane+, Point-)
    return traction & Point                                               # [...] Scalar+ <- (Plane+, Plane+, Point-, Point-)


def viscous_form(viscosity: Scalar) -> Elastic:
    """Deviatoric plane-strain shear viscosity defines a dissipation form on velocity gradients without resisting swelling."""
    return (viscosity * (shear_traction - 2 * dilation_traction / 3)) & Point  # [...] Scalar+ <- (Plane+, Plane+, Point-, Point-)

In [ ]:
def assembled_stiffness(topology: Topology, gradients: Plane, areas: Scalar, material: Elastic) -> Stiffness:
    """Pullback of a triangle work form along corner incidences assembles the nodal operator (`(Force+ <- Point-)[V <- V]`)."""
    work = material(gradients[:, None], gradients[None, :], Point, Point)
    local = (Force & Point).solve(work) * areas                           # [3, 3] (Force+ <- Point-)[F]
    corners = topology.incidence[2, 0]                                    # [3] Scalar+[F <- V]
    return (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)


def vertex_masses(topology: Topology, areas: Scalar, density: Scalar) -> Scalar:
    """Each triangle shares one third of its mass with each of its three corners (`Scalar+[V]`)."""
    corner_mean = topology.incidence[2, 0].mean(axis=0)                   # Scalar+[F <- V]
    return corner_mean.adjugate() * (density * areas)                     # Scalar+[V]


stiffness = assembled_stiffness(topology, gradients, areas, constitutive_form(lame_lambda, lame_mu))
damping = assembled_stiffness(topology, gradients, areas, viscous_form(viscosity_field))
masses = vertex_masses(topology, areas, density)                          # Scalar+[V]
inverse_inertia = (1 / masses) * Force.dual()                             # (Point- <- Force+)[V]
acceleration_map = -inverse_inertia(stiffness)                            # (Point- <- Point-)[V <- V]
drag_map = -inverse_inertia(damping)                                      # (Point- <- Point-)[V <- V]

## Oblique impact and wave propagation

A small moon strikes the upper-left rim at an oblique angle and rebounds with partial normal restitution and tangential sliding. Its lost momentum sets a compactly tapered velocity bump several triangles wide just below the surface, exciting seismic waves without putting energy into triangle-scale wavelengths that would disperse on the triangulation. Applying the elastic response to `displacement + damping_time * velocity` damps high-frequency modes in proportion to the square of their frequency while leaving long-period body and surface waves nearly intact.

The time stepping and dual-panel rendering cell below takes roughly ten seconds.

In [ ]:
impact_rotor = (mv.xy * (-scenarios.impact_angle / 2)).exp()              # [] Rotor+
impact_normal = impact_rotor >> mv.y.dual()                               # [] Point-
approach_rotor = (mv.xy * (scenarios.approach_angle / 2)).exp()           # [] Rotor+
incoming = scenarios.moon_speed * (approach_rotor >> -impact_normal)      # [] Point-
normal_part = impact_normal * (incoming | impact_normal)                  # [] Point-
outgoing = scenarios.sliding * (incoming - normal_part) - scenarios.restitution * normal_part  # [] Point-
# Momentum lost by the rebounding moon sets the impulse direction.
blow = (incoming - outgoing).normalized()                                 # [] Point-
contact = impact_normal * (scenarios.surface_radius + scenarios.moon_radius)  # [] Point-
n_approach_frames, n_frames, n_substeps = scenarios.n_approach_frames, scenarios.n_frames, scenarios.n_substeps
times = n_substeps * scenarios.dt * np.arange(-n_approach_frames, n_frames + 1)  # [n_approach_frames + n_frames + 1]
moon = contact + where(times < 0, incoming * times, outgoing * times)     # [n_approach_frames + n_frames + 1] Point-

source_centre = impact_normal * (scenarios.surface_radius - scenarios.source_depth)  # [] Point-
offsets = vertices - source_centre                                        # Point-[V]
source_radius_squared = offsets.scalar_norm_squared() / scenarios.source_width**2  # Scalar+[V]
taper = (1 - source_radius_squared / scenarios.source_support**2).squared()  # Scalar+[V]
gaussian = (-source_radius_squared / 2).exp()                             # Scalar+[V]
envelope = where(source_radius_squared < scenarios.source_support**2, gaussian * taper, 0 * gaussian)  # Scalar+[V]
velocity = scenarios.source_speed * envelope * blow                       # Point-[V]
displacement = 0 * velocity                                               # Point-[V]

In [ ]:
def divergence_operator(topology: Topology, gradients: Plane) -> Divergence:
    """Pairing each corner's simplex gradient with its nodal vector sums to the triangle divergence (`(Scalar+ <- Point-)[F <- V]`)."""
    corners = topology.incidence[2, 0]                                    # [3] Scalar+[F <- V]
    return (spdiag(gradients & Point) * corners).sum(axis=0)              # (Scalar+ <- Point-)[F <- V]


def rotation_operator(topology: Topology, gradients: Plane) -> Rotation:
    """Wedging each corner's simplex gradient with the dual inverse of its nodal vector gives the triangle rotation (`(Bivector+ <- Point-)[F <- V]`)."""
    corners = topology.incidence[2, 0]                                    # [3] Scalar+[F <- V]
    return (spdiag(gradients ^ Point.dual_inverse()) * corners).sum(axis=0)  # (Bivector+ <- Point-)[F <- V]


divergence = divergence_operator(topology, gradients)                     # (Scalar+ <- Point-)[F <- V]
rotation = rotation_operator(topology, gradients)                         # (Bivector+ <- Point-)[F <- V]

In [ ]:
def wave_evolution(acceleration_map: Acceleration, drag_map: Acceleration, displacement: Point, velocity: Point,
                   damping_time: float, dt: float, n_frames: int, n_substeps: int) -> Iterator[tuple[Point, Point]]:
    """Yields nodal `(displacement, velocity)` fields (`Point-[V]`) advanced by explicit viscoelastic velocity Verlet."""
    acceleration = acceleration_map(displacement + damping_time * velocity) + drag_map(velocity)  # Point-[V]
    yield displacement, velocity
    for _ in range(n_frames):
        for _ in range(n_substeps):
            velocity = velocity + (dt / 2) * acceleration
            displacement = displacement + dt * velocity
            # Reading strain rate from the half-step velocity keeps the time step explicit.
            acceleration = acceleration_map(displacement + damping_time * velocity) + drag_map(velocity)
            velocity = velocity + (dt / 2) * acceleration
        yield displacement, velocity


def wave_readings(divergence: Divergence, rotation: Rotation,
                  states: Iterable[tuple[Point, Point]]) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """Yields `(displacement, dilation_rate, vorticity)` fields on each frame of `states`."""
    for displacement, velocity in states:
        dilation_rate = divergence(velocity)                              # Scalar+[F]
        vorticity = rotation(velocity) & antiscalar                       # Scalar+[F]
        yield displacement, dilation_rate, vorticity


recorded_states: list[tuple[Point, Point]] = []
evolution = (recorded_states.append(state) or state
             for state in wave_evolution(acceleration_map, drag_map, displacement, velocity,
                                         scenarios.damping_time, scenarios.dt, n_frames, n_substeps))
full_states = chain(repeat((displacement, displacement), n_approach_frames), evolution)
planet_frames = show_states(
    wave_readings(divergence, rotation, full_states),
    render.waves_animation,
    topology, vertices, regions, moon, scenarios.moon_radius,
    scenarios.displacement_scale, scenarios.dilation_scale, scenarios.vorticity_scale,
    name="seismic_waves_planet",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=n_approach_frames + n_frames + 1,
);

The left panel colours dilation rate from red in compression to blue in expansion, and the right panel colours vorticity from purple for counter-clockwise turning to green for clockwise turning, each on a fixed scale. The P wave front leads on the left and refracts into a focus inside the core, while the S wave front follows on the right and halts at the core's shear shadow; behind both, the Rayleigh surface wave carries strong paired crests around the rim to meet on the antipode as exaggerated surface markers trace retrograde particle loops.

## Normal modes and travelling waves

Free oscillations keep their spatial shape while swinging harmonically in amplitude as generalized eigenmodes of the stiffness against the vertex masses. On a uniform disk of cork with zero first Lamé parameter, the P-wave speed exceeds the S-wave speed only by the square root of two, so low-order modes visibly mix compression and shear, and rotational symmetry pairs modes of equal frequency that combine at a quarter-cycle phase offset into waves travelling around the rim.

Computing the modes and rendering the twelve travelling-wave cycles below takes roughly ten seconds.

In matrix structural-dynamics notation, the free modes solve $K\boldsymbol\phi = \omega^2 M\boldsymbol\phi$, and a degenerate pair forms the travelling wave $\cos(\omega t)\,\boldsymbol\phi_1 + \sin(\omega t)\,\boldsymbol\phi_2$.

In [ ]:
def field_energy(masses: Scalar, stiffness: Stiffness, displacement: Point, velocity: Point) -> Scalar:
    """Kinetic and elastic energy sum across all vertices (`Scalar+`)."""
    kinetic = (masses * velocity.scalar_norm_squared()).sites.sum() / 2   # [] Scalar+
    elastic = (stiffness(displacement) & displacement).sites.sum() / 2    # [] Scalar+
    return kinetic + elastic                                              # [] Scalar+


def normal_modes(stiffness: Stiffness, masses: Scalar, n_total_modes: int) -> tuple[Scalar, Point]:
    """Generalized eigenmodes of `stiffness` against `masses` give squared frequencies (`[n_total_modes] Scalar+`) and shapes (`[n_total_modes] Point-[V]`)."""
    to_points = Force.dual()                                              # [] Point- <- Force+
    return to_points(stiffness).eigh(spdiag(masses * Point), n_total_modes)


n_rigid_motions, n_modes = scenarios.n_rigid_motions, scenarios.n_modes
ball_regions = np.ones_like(regions)                                      # [F]
ball_material = constitutive_form(mv.scalar([scenarios.ball_lambda]), mv.scalar([scenarios.ball_mu]))
ball_stiffness = assembled_stiffness(topology, gradients, areas, ball_material)  # (Force+ <- Point-)[V <- V]
ball_masses = vertex_masses(topology, areas, mv.scalar([scenarios.ball_density]))      # Scalar+[V]
squared_frequencies, mode_shapes = normal_modes(ball_stiffness, ball_masses, n_rigid_motions + n_modes)
oscillation_frequencies = squared_frequencies[n_rigid_motions:]           # [n_modes] Scalar+
oscillation_shapes = mode_shapes[n_rigid_motions:]                        # [n_modes] Point-[V]

In [ ]:
def travelling_waves(divergence: Divergence, rotation: Rotation, first_modes: Point, second_modes: Point,
                     peak: float, n_cycle_frames: int) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """Yields `(displacement, dilation, vorticity)` fields as each degenerate mode pair completes one quarter-phase travelling cycle."""
    scales = peak / first_modes.antinorm().batch().max(axis=-1)           # [...] Scalar+
    for pair in range(len(first_modes)):
        for phase in np.linspace(0, 2 * np.pi, n_cycle_frames, endpoint=False):
            displacement = scales[pair] * (np.cos(phase) * first_modes[pair] + np.sin(phase) * second_modes[pair])  # Point-[V]
            yield displacement, divergence(displacement), rotation(displacement) & antiscalar


n_travelling_waves, n_cycle_frames = scenarios.n_travelling_waves, scenarios.n_cycle_frames
frequency_gaps = oscillation_frequencies[1:] - oscillation_frequencies[:-1]
pair_indices = np.flatnonzero(frequency_gaps < scenarios.pair_tolerance * oscillation_frequencies[1:])[:n_travelling_waves]
show_animation(
    render.modes_animation(
        topology, vertices, ball_regions, scenarios.displacement_scale,
        travelling_waves(divergence, rotation, oscillation_shapes[pair_indices],
                         oscillation_shapes[pair_indices + 1], scenarios.mode_peak, n_cycle_frames),
    ),
    "seismic_waves_modes",
    scenarios.frame_duration_ms,
    n_frames=n_travelling_waves * n_cycle_frames,
);

Each degenerate pair runs through one travelling cycle in order of ascending frequency, with dilation on the left and rotation on the right normalized to the stronger reading in each frame. Whispering-gallery and Rayleigh-like modes ripple the outer rim while leaving the centre calm, whereas body-wave overtones send alternating compressional and rotational lobes across the interior.

In [ ]:
# checks
translations = 0 * vertices + mv.x.dual()                                 # Point-[V]
assert (stiffness(translations).norm() < 1e-11).all()
assert (stiffness(vertices.commutator(mv.xy)).norm() < 1e-11).all()
assert (squared_frequencies[:n_rigid_motions].abs() < 1e-10).all()
assert (oscillation_frequencies > 0).all()

energies = stack(tuple(field_energy(masses, stiffness, d, v) for d, v in recorded_states))  # [n_frames + 1] Scalar+
# Viscoelasticity and core viscosity dissipate energy monotonically while retaining over half the initial wave energy.
assert ((energies[1:] - energies[:-1]) < 0).all()
assert ((energies[-1] / energies[0]) > 0.5).all()

# Internal elastic and viscous forces conserve total linear and angular momentum.
initial_momentum = (masses * velocity).sites.sum()                        # [] Point-
initial_angular_momentum = (vertices & (masses * velocity)).sites.sum()   # [] Patch-
_, final_velocity = recorded_states[-1]
assert (((masses * final_velocity).sites.sum() - initial_momentum).antinorm() < 1e-11).all()
assert (((vertices & (masses * final_velocity)).sites.sum() - initial_angular_momentum).antinorm() < 1e-11).all()